In [2]:
import pandas as pd

# 1. Load one sms-call-internet-mi-YYYY-MM-DD.csv file with pandas and inspect shape, raw columns and dtypes.

df = pd.read_csv("../../data/sms-call-internet-mi-2013-11-01.csv")

print("Shape:")
print(df.shape)

print("\nRaw Columns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

Shape:
(1891928, 8)

Raw Columns:
Index(['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin',
       'callout', 'internet'],
      dtype='str')

Data Types:
datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object


In [3]:
# Find a grid and hour that has more than one country-code row
example = rows_per_grid_hour[rows_per_grid_hour["country_code_rows"] > 1].iloc[0]

example_datetime = example["datetime"]
example_cellid = example["CellID"]

print("Example datetime:", example_datetime)
print("Example CellID:", example_cellid)

example_rows = canonical_df[
    (canonical_df["datetime"] == example_datetime) &
    (canonical_df["CellID"] == example_cellid)
]

print("\nCountry-code rows for this grid and hour:")
print(example_rows[["datetime", "CellID", "countrycode"]])

NameError: name 'rows_per_grid_hour' is not defined

In [ ]:
df.head()

,datetime,CellID,countrycode,smsin,smsout,callin,callout,internet
0,2013-11-01 00:00:00,1,0,0.3521,NaN,NaN,0.0273,NaN
1,2013-11-01 00:00:00,1,33,NaN,NaN,NaN,NaN,0.0261
2,2013-11-01 00:00:00,1,39,1.7322,1.1047,0.5919,0.4020,57.7729
3,2013-11-01 00:00:00,2,0,0.3581,NaN,NaN,0.0273,NaN
4,2013-11-01 00:00:00,2,33,NaN,NaN,NaN,NaN,0.0274


In [4]:
# 2. Document the raw-to-canonical mapping and standardize datetime, CellID, countrycode, smsin, smsout, callin, callout and internet into the canonical project schema.

canonical_df = df.copy()

canonical_df["datetime"] = pd.to_datetime(canonical_df["datetime"])
canonical_df["CellID"] = canonical_df["CellID"].astype("int64")
canonical_df["countrycode"] = canonical_df["countrycode"].astype("int64")

activity_columns = ["smsin", "smsout", "callin", "callout", "internet"]

for column in activity_columns:
    canonical_df[column] = pd.to_numeric(canonical_df[column], errors="coerce")

print("Canonical columns:")
print(canonical_df.columns.tolist())

print("\nCanonical data types:")
print(canonical_df.dtypes)

Canonical columns:
['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']

Canonical data types:
datetime       datetime64[us]
CellID                  int64
countrycode             int64
smsin                 float64
smsout                float64
callin                float64
callout               float64
internet              float64
dtype: object


In [6]:
canonical_df.head()

,datetime,CellID,countrycode,smsin,smsout,callin,callout,internet
0,2013-11-01,1,0,0.3521,NaN,NaN,0.0273,NaN
1,2013-11-01,1,33,NaN,NaN,NaN,NaN,0.0261
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729
3,2013-11-01,2,0,0.3581,NaN,NaN,0.0273,NaN
4,2013-11-01,2,33,NaN,NaN,NaN,NaN,0.0274


In [7]:
# 3. Parse the timestamp field and verify the hourly cadence in the supplied file — confirm there are exactly 24 distinct timestamps and that consecutive intervals are one hour apart. Then derive date, hour and day_of_week.

# Make sure datetime is in pandas datetime format
canonical_df["datetime"] = pd.to_datetime(canonical_df["datetime"])

# Get the distinct timestamps and sort them
timestamps = canonical_df["datetime"].drop_duplicates().sort_values()

# Check the number of distinct timestamps
print("Number of distinct timestamps:", len(timestamps))

# Calculate the difference between consecutive timestamps
intervals = timestamps.diff().dropna()

print("\nTime intervals between consecutive timestamps:")
print(intervals.value_counts())

# Check whether every interval is exactly one hour
print("\nAre all intervals one hour apart?", (intervals == pd.Timedelta(hours=1)).all())

# Derive date, hour and day_of_week
canonical_df["date"] = canonical_df["datetime"].dt.date
canonical_df["hour"] = canonical_df["datetime"].dt.hour
canonical_df["day_of_week"] = canonical_df["datetime"].dt.day_name()

print("\nDerived columns:")
print(canonical_df[["datetime", "date", "hour", "day_of_week"]].head())

Number of distinct timestamps: 24

Time intervals between consecutive timestamps:
datetime
0 days 01:00:00    23
Name: count, dtype: int64

Are all intervals one hour apart? True

Derived columns:
    datetime        date  hour day_of_week
0 2013-11-01  2013-11-01     0      Friday
1 2013-11-01  2013-11-01     0      Friday
2 2013-11-01  2013-11-01     0      Friday
3 2013-11-01  2013-11-01     0      Friday
4 2013-11-01  2013-11-01     0      Friday


In [8]:
# 4. Check for missing grid_id, missing timestamp, blank activity measures, exact duplicates and negative activity values. Keep the raw file unchanged and document the curated-layer null policy separately.

# Check for missing CellID
missing_cellid = canonical_df["CellID"].isnull().sum()

# Check for missing datetime
missing_datetime = canonical_df["datetime"].isnull().sum()

# Check for blank/missing activity measures
activity_columns = ["smsin", "smsout", "callin", "callout", "internet"]
blank_activity = canonical_df[activity_columns].isnull().sum()

# Check for exact duplicate rows
duplicate_rows = canonical_df.duplicated().sum()

# Check for negative activity values
negative_activity = (canonical_df[activity_columns] < 0).sum()

print("Missing CellID:")
print(missing_cellid)

print("\nMissing datetime:")
print(missing_datetime)

print("\nBlank activity measures:")
print(blank_activity)

print("\nExact duplicate rows:")
print(duplicate_rows)

print("\nNegative activity values:")
print(negative_activity)

Missing CellID:
0

Missing datetime:
0

Blank activity measures:
smsin       1086153
smsout      1422446
callin      1407781
callout     1037413
internet    1087074
dtype: int64

Exact duplicate rows:
0

Negative activity values:
smsin       0
smsout      0
callin      0
callout     0
internet    0
dtype: int64


In [9]:
# 5. Inspect how many country-code rows exist for the same grid and hour. Confirm the raw grain is timestamp + grid_id + country_code.

# Count the number of country-code rows for each datetime and CellID
rows_per_grid_hour = (
    canonical_df
    .groupby(["datetime", "CellID"])
    .size()
    .reset_index(name="country_code_rows")
)

print("Rows per grid and hour:")
print(rows_per_grid_hour.head(10))

print("\nDistribution of country-code rows per grid and hour:")
print(rows_per_grid_hour["country_code_rows"].value_counts().sort_index())

Rows per grid and hour:
    datetime  CellID  country_code_rows
0 2013-11-01       1                  3
1 2013-11-01       2                  3
2 2013-11-01       3                  3
3 2013-11-01       4                  3
4 2013-11-01       5                  3
5 2013-11-01       6                  3
6 2013-11-01       7                  3
7 2013-11-01       8                  3
8 2013-11-01       9                  3
9 2013-11-01      10                  3

Distribution of country-code rows per grid and hour:
country_code_rows
1      3051
2     16458
3     23240
4     24053
5     24065
      ...  
57        2
58        1
59        1
64        1
65        1
Name: count, Length: 61, dtype: int64


In [10]:
# 6. Create total_sms, total_calls and total_activity as clearly labelled derived activity measures.

canonical_df["total_sms"] = canonical_df["smsin"] + canonical_df["smsout"]

canonical_df["total_calls"] = canonical_df["callin"] + canonical_df["callout"]

canonical_df["total_activity"] = (
    canonical_df["total_sms"]
    + canonical_df["total_calls"]
    + canonical_df["internet"]
)

print(canonical_df[
    [
        "smsin",
        "smsout",
        "callin",
        "callout",
        "internet",
        "total_sms",
        "total_calls",
        "total_activity"
    ]
].head(10))

    smsin  smsout  callin  callout  internet  total_sms  total_calls  \
0  0.3521     NaN     NaN   0.0273       NaN        NaN          NaN   
1     NaN     NaN     NaN      NaN    0.0261        NaN          NaN   
2  1.7322  1.1047  0.5919   0.4020   57.7729     2.8369       0.9939   
3  0.3581     NaN     NaN   0.0273       NaN        NaN          NaN   
4     NaN     NaN     NaN      NaN    0.0274        NaN          NaN   
5  1.7334  1.0880  0.6020   0.4109   57.8875     2.8214       1.0129   
6  0.3644     NaN     NaN   0.0273       NaN        NaN          NaN   
7     NaN     NaN     NaN      NaN    0.0287        NaN          NaN   
8  1.7348  1.0701  0.6128   0.4203   58.0095     2.8049       1.0331   
9  0.3349     NaN     NaN   0.0273       NaN        NaN          NaN   

   total_activity  
0             NaN  
1             NaN  
2         61.6037  
3             NaN  
4             NaN  
5         61.7218  
6             NaN  
7             NaN  
8         61.8475  
9      

In [11]:
# 7. Compute the profiling facts: number of unique grids, time range, cadence, country-code categories, busiest hourly window, busiest grid, and null counts per column.

# Number of unique grids
unique_grids = canonical_df["CellID"].nunique()

# Time range
start_time = canonical_df["datetime"].min()
end_time = canonical_df["datetime"].max()

# Cadence
timestamps = canonical_df["datetime"].drop_duplicates().sort_values()
intervals = timestamps.diff().dropna()
cadence = intervals.mode().iloc[0]

# Country-code categories
country_codes = sorted(canonical_df["countrycode"].dropna().unique())

# Activity by hour
hourly_activity = canonical_df.groupby("hour")["total_activity"].sum()

# Busiest hourly window
busiest_hour = hourly_activity.idxmax()
busiest_hour_activity = hourly_activity.max()

# Activity by grid
grid_activity = canonical_df.groupby("CellID")["total_activity"].sum()

# Busiest grid
busiest_grid = grid_activity.idxmax()
busiest_grid_activity = grid_activity.max()

# Null counts per column
null_counts = canonical_df.isnull().sum()

print("=== DATA PROFILING FACTS ===")

print("\nNumber of unique grids:")
print(unique_grids)

print("\nTime range:")
print(start_time, "to", end_time)

print("\nCadence:")
print(cadence)

print("\nCountry-code categories:")
print(country_codes)

print("\nNumber of country-code categories:")
print(len(country_codes))

print("\nBusiest hourly window:")
print("Hour:", busiest_hour)
print("Total activity:", busiest_hour_activity)

print("\nBusiest grid:")
print("CellID:", busiest_grid)
print("Total activity:", busiest_grid_activity)

print("\nNull counts per column:")
print(null_counts)

=== DATA PROFILING FACTS ===

Number of unique grids:
10000

Time range:
2013-11-01 00:00:00 to 2013-11-01 23:00:00

Cadence:
0 days 01:00:00

Country-code categories:
[np.int64(0), np.int64(1), np.int64(7), np.int64(20), np.int64(27), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(81), np.int64(82), np.int64(84), np.int64(86), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95), np.int64(98), np.int64(211), np.int64(212), np.int64(213), np.int64(216), np.int64(218), np.int64(220), np.int64(221), np.int64(222), np.int64(223), np.int64(224), np.int64(225), np.int64(22

Data Profiling Summary — Network Analytics Team


  
• The supplied daily file contains 10,000 unique grids and covers 2013-11-01 00:00:00 to 2013-11-01 23:00:00, with a confirmed hourly cadence.  
• The raw dataset contains 246 country-code categories, and the raw grain is timestamp + CellID + countrycode, allowing multiple country-code records for the same grid and hour.  
• The busiest hourly window is hour 11:00, with total activity of 5,009,588.4424.  
• The busiest grid is CellID 5161, with total activity of 262,060.3400.  
• The dataset has no missing CellID or datetime values and no exact duplicates or negative activity values, but blank activity measures are present and require an explicit curated-layer null policy.  